## Initialisation

In [1]:
# loading all necessary libraries
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier


## Loading the data

In [2]:
# reading the main df
df = pd.read_csv('/datasets/users_behavior.csv')


## Initial assessment

In [3]:
# initial assessment of the df
df.info()
print('\n-', len(df), 'rows and', len(df.columns), 'columns.')
print('- The DataFrame has already been preprocessed.')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB

- 3214 rows and 5 columns.
- The DataFrame has already been preprocessed.


In [4]:
print(df.head())


   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0


**About the DataFrame**

The dataset has already been preprocessed. Its columns describe each customer's monthly usage of the services included in the plan, and the final column identifies the plan to which the customer subscribes.

In [5]:
features = df.drop(columns='is_ultra')
target = df['is_ultra']

## Splitting the source data

* There is no separate test set, and the source dataset contains only approximately 3,000 rows, so it will be split in a 60/20/20 ratio.
* An initial 80/20 split is followed by a 75/25 split of the 80% portion, producing the final 60/20/20 split.

In [6]:
# carve out the set (20%)
features_temp, features_test, target_temp, target_test = train_test_split(
    features, target, test_size=0.20, stratify=target, random_state=123
)

# split the rest into train (60% of total) and validation (20% of total)
features_train, features_valid, target_train, target_valid = train_test_split(
    features_temp, target_temp, test_size=0.25, stratify=target_temp, random_state=123
)

## Evaluating three classification models on the validation set

### Decision Tree Classifier

In [7]:
# defining initial evaluation parameters
best_dtc_depth_valid = 0
best_dtc_accuracy_valid = 0


In [8]:
# structuring the model and validating it with up to 10 depths
for depth in range (1, 11):
    model_dtc = DecisionTreeClassifier(random_state=123, max_depth=depth)
    model_dtc.fit(features_train, target_train)
    
    # measuring model accuracy
    accuracy_valid_dtc = model_dtc.score(features_valid, target_valid)
    print('For depth', depth, ', the accuracy was', accuracy_valid_dtc)
    
    # selecting the best accuracy and the best depth
    if accuracy_valid_dtc > best_dtc_accuracy_valid:
        best_dtc_accuracy_valid = accuracy_valid_dtc
        best_dtc_depth_valid = depth
        
print('\nThe best validation accuracy for the Decision Tree Classifier was:', 
      best_dtc_accuracy_valid, 'at depth:', best_dtc_depth_valid, '.')


For depth 1 , the accuracy was 0.7558320373250389
For depth 2 , the accuracy was 0.7729393468118196
For depth 3 , the accuracy was 0.7853810264385692
For depth 4 , the accuracy was 0.7853810264385692
For depth 5 , the accuracy was 0.7838258164852255
For depth 6 , the accuracy was 0.7776049766718507
For depth 7 , the accuracy was 0.7884914463452566
For depth 8 , the accuracy was 0.7853810264385692
For depth 9 , the accuracy was 0.7931570762052877
For depth 10 , the accuracy was 0.7807153965785381

The best validation accuracy for the Decision Tree Classifier was: 0.7931570762052877 at depth: 9 .


### Random Forest Classifier

In [9]:
# defining initial evaluation parameters
best_rfc_est_valid = 0
best_rfc_accuracy_valid = 0
best_rfc_depth = 0


In [10]:
# structuring the model and validating it with up to 100 n_estimators
for est in range(10, 101, 10):
    for depth in range(1, 11):
        model_rfc = RandomForestClassifier(random_state=123, n_estimators=est, max_depth=depth)
        model_rfc.fit(features_train, target_train)

        # measuring model accuracy
        accuracy_valid_rfc = model_rfc.score(features_valid, target_valid)
        print('For n_estimators =', est, 'with depth', depth, ', the accuracy was', accuracy_valid_rfc)
    
        # selecting the best accuracy and the best est
        if accuracy_valid_rfc > best_rfc_accuracy_valid:
            best_rfc_accuracy_valid = accuracy_valid_rfc
            best_rfc_est_valid = est
            best_rfc_depth = depth


For n_estimators = 10 with depth 1 , the accuracy was 0.7573872472783826
For n_estimators = 10 with depth 2 , the accuracy was 0.7776049766718507
For n_estimators = 10 with depth 3 , the accuracy was 0.7838258164852255
For n_estimators = 10 with depth 4 , the accuracy was 0.7947122861586314
For n_estimators = 10 with depth 5 , the accuracy was 0.7884914463452566
For n_estimators = 10 with depth 6 , the accuracy was 0.7931570762052877
For n_estimators = 10 with depth 7 , the accuracy was 0.7947122861586314
For n_estimators = 10 with depth 8 , the accuracy was 0.7916018662519441
For n_estimators = 10 with depth 9 , the accuracy was 0.7947122861586314
For n_estimators = 10 with depth 10 , the accuracy was 0.7978227060653188
For n_estimators = 20 with depth 1 , the accuracy was 0.776049766718507
For n_estimators = 20 with depth 2 , the accuracy was 0.7713841368584758
For n_estimators = 20 with depth 3 , the accuracy was 0.7853810264385692
For n_estimators = 20 with depth 4 , the accuracy w

In [11]:
print('The best validation accuracy for the Random Forest Classifier was', 
      best_rfc_accuracy_valid, 'with', best_rfc_est_valid, 'estimators and a maximum depth of', 
      best_rfc_depth, '.')

The best validation accuracy for the Random Forest Classifier was 0.8040435458786936 with 20 estimators and a maximum depth of 10.


### Logistic Regression

In [12]:
# structuring the model and validating it
model_lr = LogisticRegression(random_state=123, solver='liblinear')
model_lr.fit(features_train, target_train)

# measuring model accuracy
accuracy_valid_lr = model_lr.score(features_valid, target_valid).round(4)

print('The validation accuracy for Logistic Regression was', accuracy_valid_lr)

The validation accuracy for Logistic Regression was 0.7512


**Validation results for the three models**

**Decision Tree Classifier:**
The best accuracy for this model was 0.7932 at depth 9. Beyond this depth, the model tends not to achieve meaningful improvements in accuracy, while computational cost increases.

**Random Forest Classifier:**
The best accuracy for this model was 0.804 with 20 estimators and a maximum depth of 10. As with the Decision Tree Classifier, increasing the number of estimators and searching across multiple depths increases computational cost without a substantial improvement in accuracy.

**Logistic Regression:**
This model achieved an accuracy of 0.7512, just above the project's minimum requirement.

## Evaluating model performance on the test set

### Decision Tree Classifier

* Retraining the best model identified during validation

In [13]:
# training the model using the validated hyperparameters
model_dtc = DecisionTreeClassifier(random_state=123, max_depth=9)
model_dtc.fit(features_train, target_train)


DecisionTreeClassifier(max_depth=9, random_state=123)

In [14]:

# applying the test dataset to the model
test_predictins_dtc = model_dtc.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_dtc = accuracy_score(target_test, test_predictins_dtc).round(4)

print('The test accuracy for the Decision Tree Classifier was', 
      accuracy_test_dtc)



The test accuracy for the Decision Tree Classifier was 0.7947


### Random Forest Classifier

* Applying the test data without retraining

In [15]:
# applying the test dataset to the model
test_predictions_rfc = model_rfc.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_rfc = accuracy_score(target_test, test_predictions_rfc).round(4)

print('The test accuracy for the Random Forest Classifier was',
     accuracy_test_rfc)


The test accuracy for the Random Forest Classifier was 0.8118


* Evaluating the model after retraining with the best configuration identified during validation: `n_estimators=20`

In [16]:
# training the model using the validated hyperparameters
model_rfc = RandomForestClassifier(random_state=123, n_estimators=20, max_depth=10)
model_rfc.fit(features_train, target_train)

RandomForestClassifier(max_depth=10, n_estimators=20, random_state=123)

In [17]:

# applying the test dataset to the model
test_predictions_rfc = model_rfc.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_rfc = accuracy_score(target_test, test_predictions_rfc).round(4)

print('The test accuracy for the Random Forest Classifier was',
     accuracy_test_rfc)



The test accuracy for the Random Forest Classifier was 0.8025


* Evaluating the model with the final hyperparameter value: `n_estimators=100`

In [18]:
# training the model using the validated hyperparameters
model_rfc = RandomForestClassifier(random_state=123, n_estimators=100, max_depth=10)
model_rfc.fit(features_train, target_train)

RandomForestClassifier(max_depth=10, random_state=123)

In [19]:
# applying the test dataset to the model
test_predictions_rfc = model_rfc.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_rfc = accuracy_score(target_test, test_predictions_rfc).round(4)

print('The test accuracy for the Random Forest Classifier was',
     accuracy_test_rfc)



The test accuracy for the Random Forest Classifier was 0.8118


### Logistic Regression

In [20]:
# applying the test dataset to the model
test_predictions_lr = model_lr.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_rl = accuracy_score(target_test, test_predictions_lr).round(4)

print('The test accuracy for Logistic Regression was',
     accuracy_test_rl)


The test accuracy for Logistic Regression was 0.7683


**Model results on the test set**

**Decision Tree Classifier:**
The model achieved a test accuracy of 0.7947. This is very close to its validation accuracy of 0.7932 and indicates good, consistent performance.

**Random Forest Classifier:**
The model achieved a test accuracy of 0.8118 before retraining with the best hyperparameters and 0.8025 after retraining with those hyperparameters. As with the Decision Tree Classifier, test accuracy was close to validation accuracy (0.804). This model therefore also shows good, consistent performance.

**Logistic Regression:**
The model achieved a test accuracy of 0.7683. This is close to its validation accuracy of 0.7512 and exceeds the project's minimum requirement. However, it performs less well than the other available models, so it should not be selected for this project.

The Decision Tree and Random Forest models achieved very similar test accuracies. Both exceeded the required accuracy on the validation and test sets, with similar scores, so selecting the Decision Tree model is a reasonable option in this case.

In addition to theoretically requiring more computational resources, the Random Forest model took longer to train. When considering scalability, a larger dataset would result in longer processing times for Random Forest. Choosing a model that meets the requirements in less time, even with more data, is therefore the more reasonable option.

## Comparing the models with a baseline

* Assessing how a majority-class baseline would perform using the original data

In [21]:
dumb_accuracy = df['is_ultra'].value_counts(
    normalize=True).round(2).reset_index().rename(columns={'index': 'is_ultra',
                                                          'is_ultra': 'percent'})
print(dumb_accuracy, '\n')
print('The majority-class baseline accuracy is', dumb_accuracy.loc[0, 'percent'])

   is_ultra  percent
0         0     0.69
1         1     0.31 

The majority-class baseline accuracy is 0.69


In [22]:
# applying dummy model
dummy = DummyClassifier(strategy='most_frequent', random_state=123)
dummy.fit(features_train, target_train)
dummy_accuracy = dummy.score(features_test, target_test)

print('The dummy model accuracy was', dummy_accuracy)

The dummy model accuracy was 0.6936236391912908


**Evaluating the model results**

Compared with the baseline model (DummyClassifier), **Logistic Regression** performed better: its test accuracy was 76.83%, compared with 69.36% for the baseline, a difference of 7.47 percentage points. Its validation accuracy was 75.12%. Logistic regression therefore exceeds the minimum requirement of 75% on both sets and should not be discarded on the grounds that its performance is equivalent to the baseline. However, among the configurations evaluated, it achieved lower accuracy than the Decision Tree and Random Forest models.

Both the **Decision Tree Classifier** and the **Random Forest Classifier** performed approximately 15% better than the baseline model and consistently exceeded the project's accuracy threshold of 0.75. These models can therefore be considered suitable for the project, and adding more test data and subsequently adjusting the hyperparameters could lead to even better results in the future.